# 02 Feature Engineering (Advanced)
Transforming sensor data with rolling windows, lags, and FFT features.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Add src to path
sys.path.append('..')
from src.features import engineer_features

print('Functions imported successfully!')

## 1. Load Data

In [ ]:
cols = ['unit', 'cycle', 'setting1', 'setting2', 'setting3'] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv('../data/raw/train_FD001.txt', sep=' ', header=None, names=cols)
train.dropna(axis=1, inplace=True)

# Calculate RUL
max_cycles = train.groupby('unit')['cycle'].max().reset_index()
max_cycles.columns = ['unit', 'max_cycle']
train = train.merge(max_cycles, on='unit')
train['RUL'] = train['max_cycle'] - train['cycle']
train.drop('max_cycle', axis=1, inplace=True)

print('Data loaded and RUL created. Shape:', train.shape)

## 2. Advanced Feature Engineering
Applying Variance Threshold, Rolling Stats, Lag Features, and FFT.

In [ ]:
train_processed, good_sensors = engineer_features(train)
print(f'Keeping {len(good_sensors)} sensors: {good_sensors}')
print(f'Total features after rolling, lag and FFT: {train_processed.shape[1]}')

## 3. Save Processed Features

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
feature_cols = [c for c in train_processed.columns if c not in ['unit', 'cycle', 'setting1', 'setting2', 'setting3']]
train_processed[feature_cols].to_parquet('../data/processed/features_train.parquet', index=False)
print(f'Saved {len(feature_cols)-1} features + RUL target to data/processed/features_train.parquet')